<a href="https://colab.research.google.com/github/soraiabsa/pepiline-ia/blob/main/etapa1_ofc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os
import pandas as pd
import numpy as np
from google.colab import drive

# 1. Conectar o Colab ao Google Drive
drive.mount('/content/drive')

# 2. Fixar a semente aleatória para garantir a reprodutibilidade
np.random.seed(42)

print("✅ Google Drive conectado com sucesso!")

Mounted at /content/drive
✅ Google Drive conectado com sucesso!


In [ ]:
# Definir o caminho exato da pasta onde você colocou os dados
caminho_dados = '/content/drive/MyDrive/Colab Notebooks/dados'

# Listar os arquivos presentes na pasta
arquivos = os.listdir(caminho_dados)

print("📁 Arquivos encontrados na pasta 'dados':")

for arq in arquivos:
    print(" -", arq)

📁 Arquivos encontrados na pasta 'dados':
 - TS_ALUNO_34EM (1).csv
 - TS_ALUNO_34EM.csv


In [ ]:
# Exemplo de leitura rápida das 5 primeiras linhas
arquivo_exemplo = f"{caminho_dados}/TS_ALUNO_34EM.csv"

# Carregar uma amostra pequena
df_amostra = pd.read_csv(
    arquivo_exemplo,
    sep=';',
    encoding='latin-1',
    nrows=5
)

print("🔍 Primeiras 25 colunas encontradas na base:")
print(df_amostra.columns[:25].tolist())

🔍 Primeiras 25 colunas encontradas na base:
['ID_SAEB', 'ID_REGIAO', 'ID_UF', 'ID_MUNICIPIO', 'ID_AREA', 'ID_ESCOLA', 'IN_PUBLICA', 'ID_LOCALIZACAO', 'ID_TURMA', 'ID_SERIE', 'ID_ALUNO', 'IN_SITUACAO_CENSO', 'IN_PREENCHIMENTO_LP', 'IN_PREENCHIMENTO_MT', 'IN_PRESENCA_LP', 'IN_PRESENCA_MT', 'ID_CADERNO_LP', 'ID_BLOCO_1_LP', 'ID_BLOCO_2_LP', 'ID_CADERNO_MT', 'ID_BLOCO_1_MT', 'ID_BLOCO_2_MT', 'TX_RESP_BLOCO1_LP', 'TX_RESP_BLOCO2_LP', 'TX_RESP_BLOCO1_MT']


In [ ]:
# -------------------------------------------------------------------------
# CÉLULA 4: Identificar notas e agrupar alunos por escola (ID_ESCOLA)
# -------------------------------------------------------------------------

# 1. Procurar automaticamente por colunas que contêm as notas/proficiências
colunas_notas = [
    col for col in df_amostra.columns
    if 'PROFICIENCIA' in col or 'MEDIA' in col or 'NOTA' in col
]

print("🎯 Colunas de notas encontradas:", colunas_notas)

# 2. Definir o caminho completo dos arquivos
arq_2019 = f"{caminho_dados}/TS_ALUNO_34EM.csv"
arq_2023 = f"{caminho_dados}/TS_ALUNO_34EM_2023.csv"

# 3. Identificar as colunas de Língua Portuguesa e Matemática
col_lp = 'PROFICIENCIA_LP' if 'PROFICIENCIA_LP' in colunas_notas else colunas_notas[0]

col_mt = 'PROFICIENCIA_MT' if 'PROFICIENCIA_MT' in colunas_notas else colunas_notas[1]

print(f"\nUsando '{col_lp}' para Língua Portuguesa e '{col_mt}' para Matemática.")

🎯 Colunas de notas encontradas: ['IN_PROFICIENCIA_LP', 'IN_PROFICIENCIA_MT', 'PROFICIENCIA_LP', 'PROFICIENCIA_LP_SAEB', 'PROFICIENCIA_MT', 'PROFICIENCIA_MT_SAEB']

Usando 'PROFICIENCIA_LP' para Língua Portuguesa e 'PROFICIENCIA_MT' para Matemática.


In [11]:
# -------------------------------------------------------------------------
# CÉLULA 5 (CORRIGIDA): Padronização do ID_ESCOLA como Texto Limpo
# -------------------------------------------------------------------------

import pandas as pd
import numpy as np

print("Processando dados de 2019...")

df_2019_full = pd.read_csv(
    arq_2019,
    sep=';',
    decimal=',',
    encoding='latin-1',
    usecols=[
        'ID_ESCOLA',
        'IN_PUBLICA',
        'ID_LOCALIZACAO',
        'PROFICIENCIA_LP',
        'PROFICIENCIA_MT'
    ]
)

# Converter proficiências para número
df_2019_full['PROFICIENCIA_LP'] = pd.to_numeric(
    df_2019_full['PROFICIENCIA_LP'],
    errors='coerce'
)

df_2019_full['PROFICIENCIA_MT'] = pd.to_numeric(
    df_2019_full['PROFICIENCIA_MT'],
    errors='coerce'
)

# Padronizar ID_ESCOLA como texto limpo
# Remover .0 e espaços em branco
df_2019_full = df_2019_full.dropna(subset=['ID_ESCOLA'])

df_2019_full['ID_ESCOLA'] = (
    df_2019_full['ID_ESCOLA']
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.strip()
)

# Agrupar 2019 por escola
escolas_2019 = df_2019_full.groupby('ID_ESCOLA').agg({
    'IN_PUBLICA': 'first',
    'ID_LOCALIZACAO': 'first',
    'PROFICIENCIA_LP': 'mean',
    'PROFICIENCIA_MT': 'mean'
}).reset_index().rename(columns={
    'PROFICIENCIA_LP': 'MEDIA_LP_2019',
    'PROFICIENCIA_MT': 'MEDIA_MT_2019'
})

print(f"-> Escolas processadas em 2019: {len(escolas_2019)}")

print("\nProcessando dados de 2023...")

df_2023_full = pd.read_csv(
    arq_2023,
    sep=';',
    decimal=',',
    encoding='latin-1',
    usecols=[
        'ID_ESCOLA',
        'PROFICIENCIA_LP',
        'PROFICIENCIA_MT'
    ]
)

# Converter proficiências para número
df_2023_full['PROFICIENCIA_LP'] = pd.to_numeric(
    df_2023_full['PROFICIENCIA_LP'],
    errors='coerce'
)

df_2023_full['PROFICIENCIA_MT'] = pd.to_numeric(
    df_2023_full['PROFICIENCIA_MT'],
    errors='coerce'
)

# Padronizar ID_ESCOLA como texto limpo
df_2023_full = df_2023_full.dropna(subset=['ID_ESCOLA'])

df_2023_full['ID_ESCOLA'] = (
    df_2023_full['ID_ESCOLA']
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.strip()
)

# Agrupar 2023 por escola
escolas_2023 = df_2023_full.groupby('ID_ESCOLA').agg({
    'PROFICIENCIA_LP': 'mean',
    'PROFICIENCIA_MT': 'mean'
}).reset_index().rename(columns={
    'PROFICIENCIA_LP': 'MEDIA_LP_2023',
    'PROFICIENCIA_MT': 'MEDIA_MT_2023'
})

print(f"-> Escolas processadas em 2023: {len(escolas_2023)}")

# Unir as duas edições pelo ID_ESCOLA padronizado
df_final = pd.merge(
    escolas_2019,
    escolas_2023,
    on='ID_ESCOLA',
    how='inner'
)

# Remover registros sem nota média válida
df_final = df_final.dropna(
    subset=[
        'MEDIA_LP_2019',
        'MEDIA_MT_2019',
        'MEDIA_LP_2023',
        'MEDIA_MT_2023'
    ]
)

# Criar nota total de cada ano
df_final['NOTA_TOTAL_2019'] = (
    df_final['MEDIA_LP_2019'] + df_final['MEDIA_MT_2019']
)

df_final['NOTA_TOTAL_2023'] = (
    df_final['MEDIA_LP_2023'] + df_final['MEDIA_MT_2023']
)

# Criar variável-alvo binária
# 1 = Melhorou | 0 = Não melhorou
df_final['Melhorou_Desempenho'] = (
    df_final['NOTA_TOTAL_2023'] > df_final['NOTA_TOTAL_2019']
).astype(int)

# Exibir resultados
print("\n✅ Base consolidada criada com sucesso!")
print(f"Total de escolas comparadas: {len(df_final)}")

print(
    df_final[
        [
            'ID_ESCOLA',
            'NOTA_TOTAL_2019',
            'NOTA_TOTAL_2023',
            'Melhorou_Desempenho'
        ]
    ].head()
)

Processando dados de 2019...
-> Escolas processadas em 2019: 20730

Processando dados de 2023...
-> Escolas processadas em 2023: 20083

✅ Base consolidada criada com sucesso!
Total de escolas comparadas: 0
Empty DataFrame
Columns: [ID_ESCOLA, NOTA_TOTAL_2019, NOTA_TOTAL_2023, Melhorou_Desempenho]
Index: []
